# Data Preprocessing

# Cell 1: imports and load

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../data/india_housing_prices.csv")
print(df.shape)
df.head()

(250000, 23)


,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,...,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.76,0.10,1990,...,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.52,0.08,2008,...,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.79,0.05,1997,...,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.29,0.11,1991,...,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.90,0.04,2002,...,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move


# Cell 2: first look

In [2]:
df.info()
df.describe(include="all").T

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 23 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ID                              250000 non-null  int64  
 1   State                           250000 non-null  str    
 2   City                            250000 non-null  str    
 3   Locality                        250000 non-null  str    
 4   Property_Type                   250000 non-null  str    
 5   BHK                             250000 non-null  int64  
 6   Size_in_SqFt                    250000 non-null  int64  
 7   Price_in_Lakhs                  250000 non-null  float64
 8   Price_per_SqFt                  250000 non-null  float64
 9   Year_Built                      250000 non-null  int64  
 10  Furnished_Status                250000 non-null  str    
 11  Floor_No                        250000 non-null  int64  
 12  Total_Floors               

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ID,250000.0,NaN,NaN,NaN,125000.5,72168.927986,1.0,62500.75,125000.5,187500.25,250000.0
State,250000,20,Odisha,12681,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,250000,42,Coimbatore,6461,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Locality,250000,500,Locality_296,567,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Property_Type,250000,3,Villa,83744,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BHK,250000.0,NaN,NaN,NaN,2.999396,1.415521,1.0,2.0,3.0,4.0,5.0
Size_in_SqFt,250000.0,NaN,NaN,NaN,2749.813216,1300.606954,500.0,1623.0,2747.0,3874.0,5000.0
Price_in_Lakhs,250000.0,NaN,NaN,NaN,254.586854,141.349921,10.0,132.55,253.87,376.88,500.0
Price_per_SqFt,250000.0,NaN,NaN,NaN,0.130597,0.130747,0.0,0.05,0.09,0.16,0.99
Year_Built,250000.0,NaN,NaN,NaN,2006.520012,9.808575,1990.0,1998.0,2007.0,2015.0,2023.0


# Cell 3: missing values and duplicates

In [3]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["ID"].duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

Missing values:
 ID                                0
State                             0
City                              0
Locality                          0
Property_Type                     0
BHK                               0
Size_in_SqFt                      0
Price_in_Lakhs                    0
Price_per_SqFt                    0
Year_Built                        0
Furnished_Status                  0
Floor_No                          0
Total_Floors                      0
Age_of_Property                   0
Nearby_Schools                    0
Nearby_Hospitals                  0
Public_Transport_Accessibility    0
Parking_Space                     0
Security                          0
Amenities                         0
Facing                            0
Owner_Type                        0
Availability_Status               0
dtype: int64

Duplicate rows: 0
Duplicate IDs: 0


# Cell 4: impute missing values

In [4]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include="object").columns.tolist()

for c in num_cols:
    if df[c].isnull().sum() > 0:
        df[c] = df[c].fillna(df[c].median())

for c in cat_cols:
    if df[c].isnull().sum() > 0:
        df[c] = df[c].fillna(df[c].mode()[0])

print("Remaining nulls:", df.isnull().sum().sum())

Remaining nulls: 0


# Cell 5: outlier detection

In [5]:
def outlier_summary(data, cols):
    rows = []
    for c in cols:
        q1, q3 = data[c].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n = ((data[c] < lo) | (data[c] > hi)).sum()
        rows.append([c, lo, hi, n, round(n / len(data) * 100, 2)])
    return pd.DataFrame(rows, columns=["feature", "lower", "upper", "n_outliers", "pct"])

outlier_summary(df, ["Price_in_Lakhs", "Size_in_SqFt", "Price_per_SqFt"])

,feature,lower,upper,n_outliers,pct
0,Price_in_Lakhs,-233.945,743.375,0,0.00
1,Size_in_SqFt,-1753.500,7250.500,0,0.00
2,Price_per_SqFt,-0.115,0.325,20020,8.01


# Cell 6: handle outliers by capping

In [6]:
df_clean = df.copy()
for c in ["Price_in_Lakhs", "Size_in_SqFt", "Price_per_SqFt"]:
    q1, q3 = df_clean[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    df_clean[c] = df_clean[c].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# Cell 7: save the cleaned dataset

In [7]:
df_clean.to_csv("../data/cleaned_housing.csv", index=False)
df.to_csv("../data/raw_deduped_housing.csv", index=False)
print("Saved.")

Saved.


# Cell 8: inspect the categorical columns

In [8]:
cat_inspect = ["State","Property_Type","Furnished_Status","Public_Transport_Accessibility",
               "Parking_Space","Security","Amenities","Facing","Owner_Type","Availability_Status"]
for c in cat_inspect:
    print(f"--- {c} ({df_clean[c].nunique()} unique)")
    print(df_clean[c].value_counts().head(8))
    print()

print("Cities:", df_clean["City"].nunique(), "| Localities:", df_clean["Locality"].nunique())
print(df_clean[["BHK","Floor_No","Total_Floors","Age_of_Property","Nearby_Schools","Nearby_Hospitals"]].describe().T)
print((df_clean["Age_of_Property"] + df_clean["Year_Built"]).value_counts().head())

--- State (20 unique)
State
Odisha         12681
Tamil Nadu     12629
West Bengal    12622
Gujarat        12578
Delhi          12552
Telangana      12539
Maharashtra    12533
Punjab         12516
Name: count, dtype: int64

--- Property_Type (3 unique)
Property_Type
Villa                83744
Independent House    83300
Apartment            82956
Name: count, dtype: int64

--- Furnished_Status (3 unique)
Furnished_Status
Unfurnished       83408
Semi-furnished    83374
Furnished         83218
Name: count, dtype: int64

--- Public_Transport_Accessibility (3 unique)
Public_Transport_Accessibility
High      83705
Low       83287
Medium    83008
Name: count, dtype: int64

--- Parking_Space (2 unique)
Parking_Space
No     125456
Yes    124544
Name: count, dtype: int64

--- Security (2 unique)
Security
Yes    125233
No     124767
Name: count, dtype: int64

--- Amenities (325 unique)
Amenities
Pool                 10218
Clubhouse            10010
Garden               10006
Gym                   

# Cell 9: feature engineering

In [9]:
df_fe = df_clean.copy()

# 1. Price per sqft, recomputed (Lakhs -> rupees)
df_fe["Price_per_SqFt_calc"] = (df_fe["Price_in_Lakhs"] * 100000) / df_fe["Size_in_SqFt"]

# 2. Parking: works whether the column holds numbers or Yes/No
parking_num = pd.to_numeric(df_fe["Parking_Space"], errors="coerce")
if parking_num.isna().all():
    parking_num = df_fe["Parking_Space"].astype(str).str.lower().map({"yes": 1, "no": 0}).fillna(0)
df_fe["Parking_Count"] = parking_num.fillna(0)

# 3. Amenities count (comma-separated list)
df_fe["Amenities_Count"] = df_fe["Amenities"].astype(str).str.split(",").apply(len)

# 4. Transport score (High/Medium/Low -> 3/2/1)
transport_map = {"high": 3, "medium": 2, "low": 1}
df_fe["Transport_Score"] = (df_fe["Public_Transport_Accessibility"].astype(str)
                            .str.lower().map(transport_map).fillna(2))

# 5. School density score (0-1) and hospital score (0-1)
df_fe["School_Density_Score"] = df_fe["Nearby_Schools"] / df_fe["Nearby_Schools"].max()
df_fe["Hospital_Score"] = df_fe["Nearby_Hospitals"] / df_fe["Nearby_Hospitals"].max()

# 6. Infrastructure score (0-1): schools + hospitals + transport
df_fe["Infrastructure_Score"] = (df_fe["School_Density_Score"]
                                 + df_fe["Hospital_Score"]
                                 + df_fe["Transport_Score"] / 3) / 3

# 7. Floor position and size per bedroom
df_fe["Floor_Ratio"] = df_fe["Floor_No"] / df_fe["Total_Floors"].replace(0, 1)
df_fe["SqFt_per_BHK"] = df_fe["Size_in_SqFt"] / df_fe["BHK"].replace(0, 1)

# 8. Price relative to the city median
city_median_ppsf = df_fe.groupby("City")["Price_per_SqFt"].transform("median")
df_fe["PPSF_vs_City_Median"] = df_fe["Price_per_SqFt"] / city_median_ppsf

df_fe[["Price_per_SqFt","Price_per_SqFt_calc","Parking_Count","Amenities_Count",
       "Transport_Score","Infrastructure_Score","PPSF_vs_City_Median"]].head()

,Price_per_SqFt,Price_per_SqFt_calc,Parking_Count,Amenities_Count,Transport_Score,Infrastructure_Score,PPSF_vs_City_Median
0,0.10,10332.489451,0,5,3,0.766667,1.111111
1,0.08,8270.727580,0,5,1,0.411111,0.800000
2,0.05,5046.403075,1,4,1,0.677778,0.555556
3,0.11,10955.490697,1,5,3,0.733333,1.222222
4,0.04,3792.245490,0,5,1,0.544444,0.444444


# Cell 10: target 1, “Good Investment”

In [10]:
city_median_price = df_fe.groupby("City")["Price_in_Lakhs"].transform("median")

# Each rule earns 1 point
df_fe["pts_price"]   = (df_fe["Price_in_Lakhs"] <= city_median_price).astype(int)   # price <= city median
df_fe["pts_ppsf"]    = (df_fe["Price_per_SqFt"] <= city_median_ppsf).astype(int)     # cheaper per sqft than city median
df_fe["pts_bhk"]     = (df_fe["BHK"] >= 3).astype(int)                               # BHK >= 3
df_fe["pts_ready"]   = (df_fe["Availability_Status"].astype(str).str.lower()
                        .isin(["available", "ready to move", "ready_to_move"])).astype(int)  # ready-to-move
df_fe["pts_infra"]   = (df_fe["Infrastructure_Score"] >= df_fe["Infrastructure_Score"].median()).astype(int)

pts_cols = ["pts_price","pts_ppsf","pts_bhk","pts_ready","pts_infra"]
df_fe["Investment_Score"] = df_fe[pts_cols].sum(axis=1)

df_fe["Good_Investment"] = (df_fe["Investment_Score"] >= 3).astype(int)   # threshold = 3 of 5

print(df_fe["Investment_Score"].value_counts().sort_index())
print("\nClass balance:\n", df_fe["Good_Investment"].value_counts(normalize=True).round(3))

Investment_Score
0     9368
1    38135
2    66994
3    71249
4    48763
5    15491
Name: count, dtype: int64

Class balance:
 Good_Investment
1    0.542
0    0.458
Name: proportion, dtype: float64


# Cell 11: target 2, “Future Price after 5 years”

In [11]:
# Growth rate by city: 6% to 10%, ranked by the city's median price per sqft
city_rank = df_fe.groupby("City")["Price_per_SqFt"].median().rank(pct=True)
df_fe["City_Growth_Rate"] = df_fe["City"].map(6 + 4 * city_rank) / 100

# Feature-based adjustment: better infrastructure means slightly higher growth
df_fe["Growth_Rate"] = df_fe["City_Growth_Rate"] + 0.01 * df_fe["Infrastructure_Score"]
df_fe["Growth_Rate"] = df_fe["Growth_Rate"].clip(0.05, 0.12)

YEARS = 5
df_fe["Future_Price_5Y"] = df_fe["Price_in_Lakhs"] * (1 + df_fe["Growth_Rate"]) ** YEARS

df_fe[["City","Price_in_Lakhs","Growth_Rate","Future_Price_5Y"]].head()

,City,Price_in_Lakhs,Growth_Rate,Future_Price_5Y
0,Chennai,489.76,0.087190,743.894776
1,Pune,195.52,0.103635,320.124090
2,Ludhiana,183.79,0.086302,278.018662
3,Jodhpur,300.29,0.086857,455.410657
4,Jaipur,182.90,0.084968,274.978577


# Cell 12: save the processed dataset

In [12]:
df_fe.to_csv("../data/processed_housing.csv", index=False)
print(df_fe.shape)
df_fe.columns.tolist()

(250000, 43)


['ID',
 'State',
 'City',
 'Locality',
 'Property_Type',
 'BHK',
 'Size_in_SqFt',
 'Price_in_Lakhs',
 'Price_per_SqFt',
 'Year_Built',
 'Furnished_Status',
 'Floor_No',
 'Total_Floors',
 'Age_of_Property',
 'Nearby_Schools',
 'Nearby_Hospitals',
 'Public_Transport_Accessibility',
 'Parking_Space',
 'Security',
 'Amenities',
 'Facing',
 'Owner_Type',
 'Availability_Status',
 'Price_per_SqFt_calc',
 'Parking_Count',
 'Amenities_Count',
 'Transport_Score',
 'School_Density_Score',
 'Hospital_Score',
 'Infrastructure_Score',
 'Floor_Ratio',
 'SqFt_per_BHK',
 'PPSF_vs_City_Median',
 'pts_price',
 'pts_ppsf',
 'pts_bhk',
 'pts_ready',
 'pts_infra',
 'Investment_Score',
 'Good_Investment',
 'City_Growth_Rate',
 'Growth_Rate',
 'Future_Price_5Y']

# Cell 13: rebuild with the finer price-per-sqft

In [13]:
# Use the precise price per sqft (Rs) for the median comparison
city_median_ppsf = df_fe.groupby("City")["Price_per_SqFt_calc"].transform("median")
df_fe["PPSF_vs_City_Median"] = df_fe["Price_per_SqFt_calc"] / city_median_ppsf
df_fe["pts_ppsf"] = (df_fe["Price_per_SqFt_calc"] <= city_median_ppsf).astype(int)

# Re-score the Good Investment label
pts_cols = ["pts_price","pts_ppsf","pts_bhk","pts_ready","pts_infra"]
df_fe["Investment_Score"] = df_fe[pts_cols].sum(axis=1)
df_fe["Good_Investment"] = (df_fe["Investment_Score"] >= 3).astype(int)

# Re-compute growth using the precise column
city_rank = df_fe.groupby("City")["Price_per_SqFt_calc"].median().rank(pct=True)
df_fe["City_Growth_Rate"] = df_fe["City"].map(6 + 4 * city_rank) / 100
df_fe["Growth_Rate"] = (df_fe["City_Growth_Rate"] + 0.01 * df_fe["Infrastructure_Score"]).clip(0.05, 0.12)
df_fe["Future_Price_5Y"] = df_fe["Price_in_Lakhs"] * (1 + df_fe["Growth_Rate"]) ** 5

print(df_fe["Good_Investment"].value_counts(normalize=True).round(3))
df_fe.to_csv("../data/processed_housing.csv", index=False)

Good_Investment
1    0.536
0    0.464
Name: proportion, dtype: float64
